In [ ]:
import geopandas, torch, torch_geometric, shapely, matplotlib, scipy, sklearn, networkx, shapefile
print("✅ All packages imported successfully!")

In [ ]:
import shapefile
import geopandas as gpd
import numpy as np

from matplotlib import pyplot as plt
from matplotlib.cm import get_cmap

from scipy.ndimage import gaussian_filter1d
from sklearn.preprocessing import MinMaxScaler
from scipy.interpolate import splprep, splev
from shapely.geometry import LineString
from scipy.interpolate import interp1d

import torch
from torch_geometric.data import Data
from scipy.spatial import Delaunay

import networkx as nx
from torch_geometric.utils import to_networkx

from shapely.geometry import shape as shapely_shape

## Load Input Data

In [ ]:
PATH_SHP = '../data/waterways_merged_reprojected.shp'

# Load as GeoDataFrame
gdf = gpd.read_file(PATH_SHP)
gdf = gdf.to_crs(epsg=3857)  # project to metric (optional, useful for distance ops)
print(gdf.shape)

In [ ]:
synthetic_1 = np.load(f'../data/final_dataset/sequence/real-world-river-displaced_close.npy')
synthetic_2 = np.load(f'../data/final_dataset/sequence/real-world-river-displaced_far.npy')
original = np.load(f'../data/final_dataset/sequence//real-world-river.npy')[:15000]

In [ ]:
# Create data structure of the three lines
lines_merged = [original[:500], synthetic_1[:500], synthetic_2[:500]]

In [ ]:
def make_graphs(lines):
    """
    lines: [normalized_original, normalized_synthetic_1, normalized_synthetic_2]
    Goal: predict shift from synthetic_1 -> synthetic_2
    """
    graphs = []
    num_lines = len(lines)
    num_seqs, seq_len, _ = lines[0].shape

    for seq_idx in range(num_seqs):  # iterate over sequences
        node_features_list = []
        edge_list = []

        for line_id, line_array in enumerate(lines):
            coords = line_array[seq_idx]  # (seq_len, 2)

            # Compute distances to the two other lines (for features)
            other_coords = [lines[i][seq_idx] for i in range(num_lines) if i != line_id]
            d1 = np.linalg.norm(coords - other_coords[0], axis=1, keepdims=True)
            d2 = np.linalg.norm(coords - other_coords[1], axis=1, keepdims=True)

            # Node features: [line_id, x, y, seq_len, dist1, dist2]
            line_ids = np.full((seq_len, 1), line_id)
            seq_lengths = np.full((seq_len, 1), seq_len)
            node_features = np.hstack([line_ids, coords, seq_lengths, d1, d2])
            node_features_list.append(node_features)

            # --- Within-line edges ---
            start_idx = line_id * seq_len
            src = np.arange(start_idx, start_idx + seq_len - 1)
            dst = np.arange(start_idx + 1, start_idx + seq_len)
            edges = np.vstack([np.hstack([src, dst]), np.hstack([dst, src])])
            edge_list.append(edges)

        # Cross-line edges (connect same index across lines)
        for i in range(seq_len):
            for l1 in range(num_lines):
                for l2 in range(l1 + 1, num_lines):
                    n1 = l1 * seq_len + i
                    n2 = l2 * seq_len + i
                    edge_list.append(np.array([[n1, n2], [n2, n1]]))

        # Stack features and edges
        node_features = np.vstack(node_features_list)  # (192, 6)
        edge_index = np.hstack(edge_list)              # shape (2, num_edges)

        # === TARGET: shift only for synthetic_1 nodes ===
        syn1 = lines[1][seq_idx]  # input
        syn2 = lines[2][seq_idx]  # desired output
        shift = syn2 - syn1       # Δx, Δy for each point in synthetic_1

        # Create y: zeros for other lines, shift for synthetic_1
        y = np.zeros((num_lines * seq_len, 2), dtype=np.float32)
        start_idx = 1 * seq_len  # synthetic_1 block
        y[start_idx:start_idx+seq_len] = shift

        # Convert to PyTorch tensors
        x = torch.tensor(node_features, dtype=torch.float)
        edge_index = torch.tensor(edge_index, dtype=torch.long)
        y = torch.tensor(y, dtype=torch.float)

        data = Data(x=x, edge_index=edge_index, y=y)
        graphs.append(data)

    return graphs


In [ ]:
graphs_seq = make_graphs(lines_merged)

In [ ]:
def make_graph_delaunay(lines, seq_idx):
    """
    Build one graph from all 3 lines at given sequence index,
    using Delaunay triangulation for edges.
    """
    num_lines = len(lines)
    seq_len = lines[0].shape[1]

    # Stack all points (192,2)
    coords = np.vstack([lines[i][seq_idx] for i in range(num_lines)])

    # Line IDs
    line_ids = np.repeat(np.arange(num_lines), seq_len).reshape(-1, 1)
    seq_lengths = np.full((coords.shape[0], 1), seq_len)

    # Distances to other lines (same index only, still makes sense)
    dists = []
    for li in range(num_lines):
        # compute distance to each other line
        node_coords = lines[li][seq_idx]
        others = [lines[j][seq_idx] for j in range(num_lines) if j != li]
        d1 = np.linalg.norm(node_coords - others[0], axis=1, keepdims=True)
        d2 = np.linalg.norm(node_coords - others[1], axis=1, keepdims=True)
        dists.append(np.hstack([d1, d2]))
    dists = np.vstack(dists)

    # Node features: [line_id, x, y, seq_len, dist1, dist2]
    node_features = np.hstack([line_ids, coords, seq_lengths, dists])
    x = torch.tensor(node_features, dtype=torch.float)

    # --- Delaunay triangulation edges ---
    tri = Delaunay(coords)
    edges = set()
    for simplex in tri.simplices:
        for i in range(3):
            for j in range(i+1, 3):
                edges.add((simplex[i], simplex[j]))
                edges.add((simplex[j], simplex[i]))  # undirected
    edge_index = torch.tensor(list(zip(*edges)), dtype=torch.long)

    return Data(x=x, edge_index=edge_index)

In [ ]:
graph_delaunay = make_graph_delaunay(lines_merged, seq_idx=1) #TBC it should make 

### Save Results

In [ ]:
torch.save(graphs_seq, f'../data/final_dataset/graph/graphs_sequential.pt')
torch.save(graph_delaunay, f'../data/final_dataset/graph//graph_delaunay.pt')

### Ploting Graphs

In [ ]:
def plot_graph(data):
    # Convert PyG Data -> NetworkX
    G = to_networkx(data, to_undirected=True)

    # Extract node positions (x,y from features)
    pos = {i: (float(data.x[i][1]), float(data.x[i][2])) for i in range(data.num_nodes)}

    # Node colors by line_id
    colors = [int(data.x[i][0].item()) for i in range(data.num_nodes)]

    plt.figure(figsize=(8, 6))
    nx.draw(G, pos,
            node_size=10,
            node_color=colors,
            cmap=plt.cm.Set1,
            edge_color="lightgray",
            alpha=0.8)
    plt.show()

In [ ]:
graph = graphs_seq
print(len(graph))   # 15000 graphs (one per sequence)
print(graph[0])     # first graph (192 nodes, features, edges)
print(graph[0].x.shape)        # (192, 6)
print(graph[0].edge_index.shape)  # (2, num_edges)

In [ ]:
print(graph)

In [ ]:
print(graph_delaunay)

In [ ]:
graph = graphs_seq[6] 
plot_graph(graph)